# CLUTRR Label-Free Knockoff+ FDR Gate — Calibration Diagonal, Power & Entrapment

This notebook is a **runnable demo** of the experiment
*"CLUTRR Label-Free Knockoff+ FDR Gate: Calibration Diagonal, Power & Entrapment."*

**The research question.** When an LLM extracts facts from text to feed a symbolic
(logic) layer, some extracted facts are hallucinations. Can a *label-free* admission gate
**control the false-discovery rate (FDR)** of the facts it lets through — *without* access
to any gold labels at decision time?

**The method (decoy-competition / knockoff+).** For every extracted candidate fact the
pipeline manufactures a *property-matched counterfactual decoy* and scores both with the
same provenance-blinded LLM elicitation. The signed competition statistic
`W = sign(Z_real − Z_decoy) · max(Z_real, Z_decoy)` feeds the **Barber–Candès knockoff+
threshold**, which admits a set whose FDR is provably controlled at a target `alpha`.
It is compared head-to-head against:
- a **PLAIN** raw-confidence threshold (the standard decoy-free foil), and
- a **SWAP-decoy** negative control (expected to be anti-conservative).

A deterministic **entrapment** estimator (`r=1`) gives an independent 3-way corroboration.

**What this demo runs.** The expensive part of the original pipeline is the LLM scoring
(≈4,300 OpenRouter calls on `gpt-4.1-nano`). Those per-item scores are *cached* in the
artifact's output. This demo **loads a curated 97-candidate subset of those cached scores**
and re-runs the **pure-math FDR-gate analysis** (`fdr_core.py` + the analysis functions of
`method.py`) **with zero LLM calls** — so it finishes in seconds on commodity hardware.

> The headline full run (1,540 candidates) returns `calibration_verdict = CONFIRMED`
> (bridge realized FDR ≈ 0.21 at `alpha = 0.3` vs the plain baseline rising 0.34→0.55).
> On the small demo subset you will see the same **qualitative** contrast — the knockoff+
> gate stays conservative while the plain baseline rises with `alpha` — though the strict
> certification verdict is computed honestly on the subset.

In [ ]:
# --- Dependencies -----------------------------------------------------------
# The analysis core uses only numpy (+ scipy for the tail KS/Mann-Whitney diagnostic)
# and matplotlib for the final figure. All three are pre-installed on Colab, so they are
# installed ONLY when running locally (behind the google.colab guard) at Colab's exact
# versions — installing them ON Colab would corrupt the pre-loaded C extensions.
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# (no Colab-absent packages are required by this demo)

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (original method.py import block, trimmed to the analysis core) -------
# The original orchestrator also imported asyncio / argparse / the LLM client + pipeline
# modules to *acquire* scores; this demo loads cached scores instead, so only the math
# stack is needed. scipy.stats is imported lazily inside control_behavior (as in the
# original). matplotlib is added for the visualization cell.
from __future__ import annotations

import json
import math
import sys

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- Data loading (GitHub URL with local fallback, Colab-compatible) ---------------
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6db730-decoy-gated-neuro-symbolic-extraction-a/main/round-2/experiment-1/demo/mini_demo_data.json"
import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["examples"]
print(f"dataset           : {data['dataset']}")
print(f"scorer model      : {data['meta']['model']}  (elicitation: {data['meta']['selected_elicitation']})")
print(f"candidates loaded : {len(examples)}")
print(f"families present  : {sorted(set(e['metadata_family'] for e in examples))}")
print(f"gold labels       : {sorted(set(e['metadata_label'] for e in examples))}")
print(f"n documents       : {len(set(e['metadata_doc_id'] for e in examples))}")
# peek at one cached candidate row
ex = examples[0]
print("\nexample candidate :", json.loads(ex["input"]))
print("  gold label      :", ex["metadata_label"])
print("  cached scores   :", {k: ex[k] for k in
      ("metadata_z_real", "metadata_z_cf_decoy", "metadata_z_swap_decoy", "metadata_z_entrapment")})

## Configuration

All tunable parameters live here. The two **demo knobs** (`B_BOOT`, `N_FALSE_MIN`) are set
to small/permissive values so the notebook runs quickly on the 97-candidate subset; their
original full-run values are shown in comments — restore them (and point `load_data` at the
full output) for a production run. The statistical constants (`ALPHA_GRID`, `TAU`, `SEED`)
and the pre-registered `ALPHA_STAR` / `PRIMARY_FAMILY` are read from the data's metadata so
the demo uses exactly the choices the pilot fixed.

In [ ]:
SEED          = data["meta"].get("seed", 20240617)          # bootstrap RNG seed
MODEL         = data["meta"].get("model", "openai/gpt-4.1-nano")
ALPHA_GRID    = data["meta"].get("alpha_grid", [0.05, 0.10, 0.20, 0.30, 0.50])  # target FDR grid
TAU           = data["meta"].get("tau", 0.05)               # calibration slack band (alpha + tau)
ALPHA_STAR    = data["meta"].get("alpha_star", 0.30)        # pre-registered primary alpha*
PRIMARY_FAMILY = data["meta"].get("primary_family", "bridge")  # populable family for disconfirmation

# --- demo knobs (scale these back up for a full run) -------------------------------
B_BOOT       = 300    # document-block bootstrap resamples   (original: 2000)
N_FALSE_MIN  = 10     # min genuine-FALSE candidates for a testable diagonal (original: 40)

print(f"alpha_grid={ALPHA_GRID}  alpha*={ALPHA_STAR}  primary_family={PRIMARY_FAMILY}")
print(f"B_BOOT={B_BOOT}  N_FALSE_MIN={N_FALSE_MIN}  tau={TAU}  seed={SEED}")

## `fdr_core.py` — the pure, API-free mathematical core

This is the artifact's `fdr_core.py`, reproduced **verbatim**. Every function is
deterministic with no network / I/O. It implements:

- the **canonical competition statistic** `W = sign(Z − Z̃)·max(Z, Z̃)`,
- the **knockoff+ threshold** (Barber–Candès eq. 1.9, with the `+1`),
- the `1/k` minimum-estimable-FDR floor,
- the **entrapment FDP estimators** (Wen et al. 2025: lower / combined / paired),
- the **document-block (cluster) bootstrap** for FDR/FDP confidence intervals,
- crisp CLUTRR gold labelling, the **PLAIN confidence-threshold baseline**, and the
  tail diagnostics (tail win-rate, tail-AUC).

After defining the functions we alias `fc = sys.modules[__name__]` so the original
`method.py` analysis code (next cells) keeps its `fc.`-qualified calls **unchanged**.

In [ ]:
# ============================================================================
# Labels
# ============================================================================
TRUE = "TRUE"
FALSE = "FALSE"
UNJUDGEABLE = "UNJUDGEABLE"


# ============================================================================
# MODULE 5 -- the canonical competition statistic + knockoff+ gate (SPEC1 A)
# ============================================================================
def w_statistic(z_real: float, z_decoy: float) -> float:
    """CANONICAL signed magnitude-max competition statistic (SPEC1 A, SPEC2 Sec 0):

        W_i = sign(Z_i - Z~_i) * max(Z_i, Z~_i)

    A large positive W => the real candidate beat its matched decoy with a high score
    (evidence of a true signal). Ties (Z_i == Z~_i) give sign 0 -> W = 0 (no evidence;
    never admitted at a positive cutoff). This is the iter-1 fix: the per-pair difference
    d_i = Z_i - Z~_i is a TAIL DIAGNOSTIC only and is NEVER passed to the gate.
    """
    zr, zd = float(z_real), float(z_decoy)
    s = (zr > zd) - (zr < zd)  # sign in {-1,0,+1}
    return float(s) * max(zr, zd)


def knockoff_plus_threshold(W, alpha: float):
    """knockoff+ admission threshold (Barber-Candes Definition 2, eq. 1.9; controls FDR exactly).

        T = min { t in {|W_i|} : (1 + #{i: W_i <= -t}) / (#{i: W_i >= t} v 1) <= alpha }
        admitted set  Shat = { i : W_i >= T }

    The +1 in the numerator is kept (Rajchert-Keich prove it is in general necessary).
    Scans candidate cutoffs over the ascending distinct POSITIVE |W| magnitudes and returns
    the smallest feasible t (the most permissive admission).

    Returns (T, admitted_indices(sorted list), fdr_hat). If no feasible cutoff: (inf, [], 1.0).
    """
    W = np.asarray(W, dtype=float)
    n = W.size
    if n == 0:
        return math.inf, [], 1.0
    cand = np.unique(np.abs(W))
    cand = cand[cand > 0.0]  # positive magnitudes only (|W|=0 candidates are never selected)
    if cand.size == 0:
        return math.inf, [], 1.0
    for t in cand:  # ascending => smallest feasible t first => most permissive
        pos = int(np.sum(W >= t))
        neg = int(np.sum(W <= -t))
        fdr_hat = (1 + neg) / max(1, pos)
        if fdr_hat <= alpha:
            admitted = sorted(int(i) for i in np.where(W >= t)[0])
            return float(t), admitted, float(fdr_hat)
    return math.inf, [], 1.0


def k_floor(alpha: float) -> int:
    """Minimum admissions needed to certify FDR<=alpha (the 1/k floor): k >= ceil(1/alpha)."""
    return int(math.ceil(1.0 / alpha))


def alpha_is_certifiable(n_max_admissible: int, alpha: float) -> bool:
    """An alpha is structurally demonstrable only if the maximum attainable #admissions
    can reach its k-floor ceil(1/alpha). Otherwise the alpha is precondition-unmet (NOT
    'confirmed by conservatism')."""
    return n_max_admissible >= k_floor(alpha)


# ============================================================================
# PLAIN confidence-threshold baseline gate (decoy-free primary foil; SPEC2 Block E)
# ============================================================================
def plain_threshold_gate(Z, alpha: float):
    """Decoy-free label-free baseline: admit the most-confident candidates until the
    *self-estimated* FDR of the admitted set (1 - mean admitted confidence) would exceed
    alpha. This is the standard 'raw LLM confidence' gate the decoy method is compared
    against -- it has NO null calibration, so its self-estimate is expected to be
    anti-conservative (overconfident) relative to the realized FDR against gold.

    Returns (threshold, admitted_indices, est_fdr_of_admitted).
    """
    Z = np.asarray(Z, dtype=float)
    n = Z.size
    if n == 0:
        return math.inf, [], 1.0
    order = np.argsort(-Z, kind="stable")  # descending confidence
    zsorted = Z[order]
    cumsum = np.cumsum(zsorted)
    best_k = 0
    best_est = 1.0
    for k in range(1, n + 1):
        est_fdr = 1.0 - cumsum[k - 1] / k  # 1 - mean confidence of the top-k admitted
        if est_fdr <= alpha:
            best_k = k
            best_est = est_fdr
    if best_k == 0:
        return math.inf, [], 1.0
    threshold = float(zsorted[best_k - 1])
    admitted = sorted(int(i) for i in order[:best_k])
    return threshold, admitted, float(best_est)


# ============================================================================
# MODULE 6 -- entrapment FDP estimators (Wen et al. 2025; SPEC1 B)
# ============================================================================
def entrapment_fdp(N_T: int, N_E: int, r: float, estimator: str = "combined",
                   paired_counts=None) -> float:
    """Entrapment-based FDP estimators (verbatim eq. numbers from SPEC1 B):

        lower    (eq.2)  = N_E / (N_T + N_E)                          # failure-only lower bound
        combined (eq.1)  = N_E * (1 + 1/r) / (N_T + N_E)             # DEFAULT upper bound
        paired   (eq.4)  = (N_E + N_{E>=s>T} + 2 N_{E>T>=s}) / (N_T + N_E)   # tighter, requires r==1
        sample   (eq.3)  = INVALID (biased) -> raises

    paired_counts (for 'paired'): {'E_ge_s_gt_T': int, 'E_gt_T_ge_s': int}.
    """
    denom = max(1, N_T + N_E)
    if estimator == "lower":
        return N_E / denom
    if estimator == "combined":
        return N_E * (1.0 + 1.0 / r) / denom
    if estimator == "sample":
        raise ValueError("entrapment 'sample' estimator (eq.3) is INVALID/biased -- never use it")
    if estimator == "paired":
        if abs(r - 1.0) > 1e-9:
            raise ValueError("paired entrapment estimator requires r == 1")
        if paired_counts is None:
            raise ValueError("paired estimator requires paired_counts")
        n_egt = int(paired_counts.get("E_ge_s_gt_T", 0))
        n_egtt = int(paired_counts.get("E_gt_T_ge_s", 0))
        return (N_E + n_egt + 2 * n_egtt) / denom
    raise ValueError(f"unknown estimator: {estimator}")


def paired_entrapment_counts(real_scores, entrapment_scores, admitted_mask_real,
                             admitted_mask_ent, s_cut: float):
    """Compute the paired-estimator auxiliary counts (eq.4) for one-to-one (r=1) pairing.

    For each (real_i, entrapment_i) pair, with operative discovery cutoff score s:
      N_E            = # entrapment items discovered (admitted)
      N_{E>=s>T}     = # discovered entrapment whose PAIRED real scored < s (real not discovered)
      N_{E>T>=s}     = # discovered entrapment whose paired real scored LOWER but is ALSO discovered
    Here 'score' is the per-item scalar Z and s_cut is the score threshold that defines discovery.
    """
    real_scores = np.asarray(real_scores, float)
    ent_scores = np.asarray(ent := entrapment_scores, float)
    am_real = np.asarray(admitted_mask_real, bool)
    am_ent = np.asarray(admitted_mask_ent, bool)
    N_E = int(np.sum(am_ent))
    n_egt = 0
    n_egtt = 0
    for i in range(len(ent_scores)):
        if not am_ent[i]:
            continue
        if not am_real[i]:
            # paired real NOT discovered (real score < s)
            n_egt += 1
        else:
            # paired real discovered too; "scored lower but still discovered"
            if ent_scores[i] > real_scores[i]:
                n_egtt += 1
    return {"E_ge_s_gt_T": n_egt, "E_gt_T_ge_s": n_egtt, "N_E": N_E}


# ============================================================================
# Crisp CLUTRR gold labelling (MODULE 0)
# ============================================================================
def gold_label(candidate: tuple, gold_true: set, covered_pairs: set) -> str:
    """Crisp gold label for an extracted candidate (h, r, t):

        TRUE         if (h,r,t) is a directly-stated atomic OR proof-path-derived bridge fact
        FALSE        if (h,t) is a COVERED pair (appears in gold) but the relation is wrong
                     (a genuine hallucination -- wrong relation on a known pair)
        UNJUDGEABLE  if (h,t) is not on any proof path -> excluded from the FDR pool (logged)

    Relations are compared lowercased; names exactly. This preserves CLUTRR crispness with
    NO homegrown rule reimplementation.
    """
    h, r, t = candidate
    key = (h, r.lower(), t)
    if key in gold_true:
        return TRUE
    if (h, t) in covered_pairs:
        return FALSE
    return UNJUDGEABLE


# ============================================================================
# MODULE 7 -- document-block (cluster) bootstrap (SPEC1 C)
# ============================================================================
def doc_block_bootstrap(per_doc_records: list, statistic_fn, B: int = 2000,
                        seed: int = 20240617, lo_pct: float = 2.5, hi_pct: float = 97.5):
    """Resample WHOLE documents with replacement (preserving within-doc dependence),
    re-run the statistic on each resample, return (point, lo, hi) percentile CI.

    per_doc_records : list (one element per document; any structure statistic_fn understands)
    statistic_fn    : maps a list-of-doc-records -> float (re-runs the WHOLE gate+stat)
    """
    rng = np.random.default_rng(seed)
    D = len(per_doc_records)
    point = float(statistic_fn(per_doc_records))
    if D == 0:
        return point, float("nan"), float("nan")
    stats = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, D, size=D)
        boot = [per_doc_records[i] for i in idx]
        stats[b] = statistic_fn(boot)
    stats = stats[~np.isnan(stats)]
    if stats.size == 0:
        return point, float("nan"), float("nan")
    lo = float(np.percentile(stats, lo_pct))
    hi = float(np.percentile(stats, hi_pct))
    return point, lo, hi


# ============================================================================
# MODULE 4/D.4 -- tail diagnostics (measurement only; NEVER consumed by the gate)
# ============================================================================
def auc(scores_pos, scores_neg) -> float:
    """AUC = P(score_pos > score_neg) via the Mann-Whitney U statistic (ties -> 0.5).
    Returns NaN if either class is empty."""
    p = np.asarray(scores_pos, float)
    n = np.asarray(scores_neg, float)
    if p.size == 0 or n.size == 0:
        return float("nan")
    allv = np.concatenate([p, n])
    order = np.argsort(allv, kind="stable")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, allv.size + 1)
    _assign_tie_ranks(allv, ranks)
    r_pos = ranks[: p.size].sum()
    u_pos = r_pos - p.size * (p.size + 1) / 2.0
    return float(u_pos / (p.size * n.size))


def _assign_tie_ranks(values: np.ndarray, ranks: np.ndarray) -> None:
    """In-place average-rank assignment for ties."""
    order = np.argsort(values, kind="stable")
    sv = values[order]
    i = 0
    n = sv.size
    while i < n:
        j = i
        while j + 1 < n and sv[j + 1] == sv[i]:
            j += 1
        if j > i:
            avg = (i + 1 + j + 1) / 2.0  # average of 1-based ranks
            for k in range(i, j + 1):
                ranks[order[k]] = avg
        else:
            ranks[order[i]] = i + 1
        i = j + 1


def tail_auc(scores, labels, tail_frac: float = 0.5) -> float:
    """AUC of TRUE vs FALSE restricted to the upper (admission) tail = the top `tail_frac`
    of items by score. Requires both classes present in the tail; else NaN."""
    s = np.asarray(scores, float)
    lab = np.asarray(labels, dtype=object)
    if s.size == 0:
        return float("nan")
    k = max(1, int(math.ceil(tail_frac * s.size)))
    tail_idx = np.argsort(-s, kind="stable")[:k]
    s_t = s[tail_idx]
    lab_t = lab[tail_idx]
    pos = s_t[lab_t == TRUE]
    neg = s_t[lab_t == FALSE]
    return auc(pos, neg)


def tail_win_rate(z_real, z_decoy, cut: float) -> float:
    """Tail-conditioned win-rate of the DECOY over its matched real, among pairs whose
    max(Z_real, Z_decoy) >= cut. For counterfactual decoys this should be ~0.5 (fair coin);
    for too-easy swap decoys it should be measurably < 0.5. Returns NaN if tail empty."""
    zr = np.asarray(z_real, float)
    zd = np.asarray(z_decoy, float)
    m = np.maximum(zr, zd)
    sel = m >= cut
    if not np.any(sel):
        return float("nan")
    zr_s, zd_s = zr[sel], zd[sel]
    wins = np.sum(zd_s > zr_s) + 0.5 * np.sum(zd_s == zr_s)
    return float(wins / sel.sum())


# Alias the current module as `fc` so the original method.py analysis code below keeps its
# `fc.`-qualified calls verbatim (fc.w_statistic, fc.knockoff_plus_threshold, fc.FALSE, ...).
fc = sys.modules[__name__]
print("fdr_core loaded; fc alias ->", fc.__name__)

## Gate statistics over scored doc-units (`method.py`, verbatim)

These are the analysis functions from `method.py`, reproduced **verbatim**. They flatten the
per-document scored units into arrays, run the knockoff+ / swap / plain gates at each target
`alpha`, and assemble the **calibration diagonal** (realized FDR vs target `alpha`) with
document-block bootstrap CIs. `populability` counts how many genuine-FALSE candidates the
gate could ever admit (the testability precondition).

In [ ]:
def _families(family):
    return ("atomic", "bridge") if family == "pooled" else (family,)


def family_arrays(scored_docs, family, decoy_key="z_cf"):
    """Flatten scored docs into arrays for a family ('atomic'/'bridge'/'pooled'). decoy_key
    selects which decoy's z to compete against ('z_cf' or 'z_swap')."""
    W, Z, lab, zdec = [], [], [], []
    for d in scored_docs:
        for fam in _families(family):
            for u in d["units"][fam]:
                if "z_real" not in u:
                    continue
                zr = u["z_real"]
                zd = u["z_cf"] if decoy_key == "z_cf" else u["z_swap"]
                W.append(fc.w_statistic(zr, zd))
                Z.append(zr)
                lab.append(u["label"])
                zdec.append(zd)
    return np.array(W), np.array(Z), np.array(lab, dtype=object), np.array(zdec)


def per_doc_family_records(scored_docs, family):
    """One record per doc: lists of (z_real, z_cf, z_swap, z_ent, label) for bootstrap."""
    recs = []
    for d in scored_docs:
        rows = []
        for fam in _families(family):
            for u in d["units"][fam]:
                if "z_real" not in u:
                    continue
                rows.append((u["z_real"], u["z_cf"], u["z_swap"], u.get("z_ent"), u["label"]))
        recs.append(rows)
    return recs


def realized_fdr_from_rows(rows_concat, alpha, which="knockoff"):
    """rows_concat: list of (z_real, z_cf, z_swap, z_ent, label). Returns realized FDR of the
    admitted set under the chosen gate."""
    if not rows_concat:
        return float("nan")
    zr = np.array([r[0] for r in rows_concat])
    zc = np.array([r[1] for r in rows_concat])
    zs = np.array([r[2] for r in rows_concat])
    lab = np.array([r[4] for r in rows_concat], dtype=object)
    if which == "knockoff":
        W = np.array([fc.w_statistic(a, b) for a, b in zip(zr, zc)])
        _, adm, _ = fc.knockoff_plus_threshold(W, alpha)
    elif which == "swap":
        W = np.array([fc.w_statistic(a, b) for a, b in zip(zr, zs)])
        _, adm, _ = fc.knockoff_plus_threshold(W, alpha)
    elif which == "plain":
        _, adm, _ = fc.plain_threshold_gate(zr, alpha)
    else:
        raise ValueError(which)
    if not adm:
        return float("nan")
    n_false = sum(1 for i in adm if lab[i] == fc.FALSE)
    return n_false / len(adm)


def _nan(x):
    return None if (x is None or (isinstance(x, float) and math.isnan(x))) else round(float(x), 6)


def diagonal_for_family(scored_docs, family):
    """Compute the full diagonal (knockoff / swap / plain) with bootstrap CIs for one family."""
    W, Z, lab, _ = family_arrays(scored_docs, family, "z_cf")
    n_pos = int(np.sum(W > 0))
    per_doc = per_doc_family_records(scored_docs, family)
    flat = [r for doc in per_doc for r in doc]
    rows = []
    for alpha in ALPHA_GRID:
        T, adm, decoy_fdr_hat = fc.knockoff_plus_threshold(W, alpha)
        realized = realized_fdr_from_rows(flat, alpha, "knockoff")
        n_adm = len(adm)
        n_false = sum(1 for i in adm if lab[i] == fc.FALSE)
        pt, lo, hi = fc.doc_block_bootstrap(
            per_doc, lambda recs: realized_fdr_from_rows([r for doc in recs for r in doc], alpha, "knockoff"),
            B=B_BOOT, seed=SEED)
        # swap control + plain baseline at matched alpha
        realized_swap = realized_fdr_from_rows(flat, alpha, "swap")
        realized_plain = realized_fdr_from_rows(flat, alpha, "plain")
        _, adm_swap, _ = fc.knockoff_plus_threshold(
            family_arrays(scored_docs, family, "z_swap")[0], alpha)
        _, adm_plain, est_plain = fc.plain_threshold_gate(Z, alpha)
        rows.append({
            "target_alpha": alpha, "realized_fdr": _nan(realized),
            "ci_low": _nan(lo), "ci_high": _nan(hi),
            "n_admitted": n_adm, "n_false": n_false,
            "decoy_fdr_hat": _nan(decoy_fdr_hat) if not math.isinf(decoy_fdr_hat) else None,
            "k_floor": fc.k_floor(alpha),
            "certified": bool(fc.alpha_is_certifiable(n_pos, alpha)),
            "swap_realized_fdr": _nan(realized_swap), "swap_n_admitted": len(adm_swap),
            "plain_realized_fdr": _nan(realized_plain), "plain_n_admitted": len(adm_plain),
            "plain_est_fdr": _nan(est_plain),
        })
    return {"rows": rows, "n_pos": n_pos, "n_total": int(W.size),
            "n_true": int(np.sum(lab == fc.TRUE)), "n_false_total": int(np.sum(lab == fc.FALSE))}


def populability(scored_docs, alpha_star):
    out = {}
    for fam in ("atomic", "bridge"):
        W, Z, lab, _ = family_arrays(scored_docs, fam, "z_cf")
        _, adm, _ = fc.knockoff_plus_threshold(W, alpha_star)
        n_false = sum(1 for i in adm if lab[i] == fc.FALSE)
        # also count at loosest alpha for total available
        _, adm_loose, _ = fc.knockoff_plus_threshold(W, 0.50)
        n_false_loose = sum(1 for i in adm_loose if lab[i] == fc.FALSE)
        out[fam] = {"n_admitted_at_alpha_star": len(adm), "n_false_at_alpha_star": n_false,
                    "n_false_total_in_family": int(np.sum(lab == fc.FALSE)),
                    "n_false_admitted_loosest": n_false_loose}
    pooled_false = out["atomic"]["n_false_total_in_family"] + out["bridge"]["n_false_total_in_family"]
    out["pooled"] = {"n_false_total": pooled_false}
    return out

## Decoy-control diagnostics, entrapment corroboration & the verdict (`method.py`, verbatim)

`control_behavior` measures whether the counterfactual decoys behave like a fair null in the
admission tail (win-rate ≈ 0.5) while the too-easy swap decoys do not. `entrapment_analysis`
runs the independent deterministic-entrapment FDP estimator (`r=1`) and checks 3-way agreement
with the gold-realized FDR and the decoy self-estimate. `disconfirmation` evaluates the single
pre-registered test at `alpha*`, and `calibration_verdict` combines them.

> The only change from the original is `B=B_BOOT` inside `entrapment_analysis`'s bootstrap
> (the original hard-codes `B=1000`) so the bootstrap budget is driven by the config cell.

In [ ]:
def control_behavior(scored_docs, family, alpha=0.30):
    W, Z, lab, zcf = family_arrays(scored_docs, family, "z_cf")
    _, _, _, zswap = family_arrays(scored_docs, family, "z_swap")
    T, adm, _ = fc.knockoff_plus_threshold(W, alpha)
    cut = T if not math.isinf(T) else float(np.percentile(np.maximum(Z, zcf), 50)) if Z.size else 0.5
    wr_cf = fc.tail_win_rate(Z, zcf, cut)
    wr_swap = fc.tail_win_rate(Z, zswap, cut)
    # tail KS / Mann-Whitney: real-FALSE vs decoy scores in the admission tail
    false_scores = Z[(lab == fc.FALSE) & (Z >= cut)]
    cf_tail = zcf[zcf >= cut]
    ks_p = mw_p = None
    try:
        from scipy import stats
        if false_scores.size > 1 and cf_tail.size > 1:
            ks_p = float(stats.ks_2samp(cf_tail, false_scores, alternative="greater").pvalue)
            mw_p = float(stats.mannwhitneyu(cf_tail, false_scores, alternative="less").pvalue)
    except Exception:
        pass
    return {"alpha_used": alpha, "cut": _nan(cut),
            "counterfactual_tail_win_rate": _nan(wr_cf),
            "swap_tail_win_rate": _nan(wr_swap),
            "ks_pvalue_cf_vs_realfalse": ks_p, "mannwhitney_pvalue": mw_p}


def entrapment_analysis(scored_docs, family, alpha):
    """At the operative cutoff for `alpha`, compute combined & paired entrapment FDP with
    doc-block CI, vs realized & decoy estimates."""
    W, Z, lab, zcf = family_arrays(scored_docs, family, "z_cf")
    T, adm, decoy_fdr_hat = fc.knockoff_plus_threshold(W, alpha)
    adm_set = set(adm)
    # entrapment items scored; "discovered" if their score would pass the same operative cutoff.
    # Use the knockoff competition for entrapment too: an entrapment is "admitted" if it beats
    # its paired real's decoy structure -> approximate by score >= cut on Z-scale.
    zent, zreal_paired, real_adm_mask, ent_adm_mask = [], [], [], []
    cut = T if not math.isinf(T) else float("inf")
    i = 0
    for d in scored_docs:
        for u in d["units"][family]:
            if "z_real" not in u or "z_ent" not in u:
                continue
            zent.append(u["z_ent"]); zreal_paired.append(u["z_real"])
            real_adm_mask.append(i in adm_set)
            ent_adm_mask.append(u["z_ent"] >= cut if not math.isinf(cut) else False)
            i += 1
    N_T = len(adm)
    N_E = int(np.sum(ent_adm_mask))
    combined = fc.entrapment_fdp(N_T, N_E, r=1.0, estimator="combined")
    pc = fc.paired_entrapment_counts(zreal_paired, zent, real_adm_mask, ent_adm_mask, cut)
    paired = fc.entrapment_fdp(N_T, N_E, r=1.0, estimator="paired", paired_counts=pc)

    # doc-block bootstrap CI for the combined FDP: resample docs, re-run gate+entrapment.
    per_doc = []
    for d in scored_docs:
        rows = [(u["z_real"], u["z_cf"], u.get("z_ent")) for fam in _families(family)
                for u in d["units"][fam] if "z_real" in u]
        per_doc.append(rows)

    def comb_stat(recs):
        flat = [r for doc in recs for r in doc]
        if not flat:
            return float("nan")
        zr = np.array([r[0] for r in flat]); zc = np.array([r[1] for r in flat])
        ze = np.array([(r[2] if r[2] is not None else 0.0) for r in flat])
        Wb = np.array([fc.w_statistic(a, b) for a, b in zip(zr, zc)])
        Tb, admb, _ = fc.knockoff_plus_threshold(Wb, alpha)
        if math.isinf(Tb):
            return float("nan")
        nt = len(admb)
        ne = int(np.sum(ze >= Tb))
        return fc.entrapment_fdp(nt, ne, r=1.0, estimator="combined")

    _, lo, hi = fc.doc_block_bootstrap(per_doc, comb_stat, B=B_BOOT, seed=SEED)
    flat_rows = [(u["z_real"], u["z_cf"], u["z_swap"], u.get("z_ent"), u["label"])
                 for d in scored_docs for fam in _families(family) for u in d["units"][fam] if "z_real" in u]
    realized = realized_fdr_from_rows(flat_rows, alpha, "knockoff")
    return {"alpha": alpha, "N_T": N_T, "N_E": N_E, "r": 1,
            "fdp_combined": _nan(combined), "fdp_combined_ci": [_nan(lo), _nan(hi)],
            "fdp_paired": _nan(paired),
            "decoy_fdr_hat": _nan(decoy_fdr_hat) if not math.isinf(decoy_fdr_hat) else None,
            "realized_fdr_gold": _nan(realized),
            "agree_realized": _agree(combined, realized), "agree_decoy": _agree(combined, decoy_fdr_hat),
            "tail_difficulty_ent_median": _nan(float(np.median(zent)) if zent else float("nan")),
            "tail_difficulty_real_median": _nan(float(np.median(zreal_paired)) if zreal_paired else float("nan"))}


def _agree(a, b, tol=0.10):
    if a is None or b is None or (isinstance(a, float) and math.isnan(a)) \
            or (isinstance(b, float) and (math.isnan(b) or math.isinf(b))):
        return None
    return bool(abs(float(a) - float(b)) <= tol)


def disconfirmation(scored_docs, family, alpha_star, pop):
    """Single pre-registered disconfirmation at alpha* on the populable family."""
    W, Z, lab, _ = family_arrays(scored_docs, family, "z_cf")
    per_doc = per_doc_family_records(scored_docs, family)
    flat = [r for doc in per_doc for r in doc]
    realized = realized_fdr_from_rows(flat, alpha_star, "knockoff")
    pt, lo, hi = fc.doc_block_bootstrap(
        per_doc, lambda recs: realized_fdr_from_rows([r for doc in recs for r in doc], alpha_star, "knockoff"),
        B=B_BOOT, seed=SEED)
    n_false_total = pop[family]["n_false_total_in_family"]
    if n_false_total < N_FALSE_MIN:
        verdict = "UNTESTABLE"
        reason = (f"populable family '{family}' has only {n_false_total} genuine FALSE candidates "
                  f"(< N_false_min={N_FALSE_MIN}); diagonal precondition unmet (NOT 'confirmed by conservatism').")
    elif realized is not None and not math.isnan(realized) and realized > alpha_star + TAU \
            and lo is not None and not math.isnan(lo) and lo > alpha_star:
        verdict = "DISCONFIRMED"
        reason = (f"realized FDR {realized:.3f} > alpha*+tau ({alpha_star}+{TAU}) AND doc-block "
                  f"CI [{lo:.3f},{hi:.3f}] lies entirely above alpha*={alpha_star}.")
    else:
        verdict = "NOT_DISCONFIRMED"
        reason = (f"realized FDR {realized} with CI [{lo},{hi}] does not exceed alpha*+tau with "
                  f"CI entirely above alpha*={alpha_star}.")
    return {"alpha_star": alpha_star, "family": family, "realized_fdr": _nan(realized),
            "ci": [_nan(lo), _nan(hi)], "tau": TAU, "verdict": verdict, "reason": reason,
            "n_false_total_in_family": n_false_total}


def calibration_verdict(bridge_diag, disconf):
    if disconf["verdict"] == "UNTESTABLE":
        return "UNTESTABLE"
    if disconf["verdict"] == "DISCONFIRMED":
        return "DISCONFIRMED"
    # CONFIRMED iff diagonal tracks within tau above the 1/k floor across certified alphas
    ok = True
    any_cert = False
    for row in bridge_diag["rows"]:
        if not row["certified"]:
            continue
        any_cert = True
        rf = row["realized_fdr"]
        if rf is None:
            continue
        if rf > row["target_alpha"] + TAU:
            ok = False
    return "CONFIRMED" if (any_cert and ok) else "INCONCLUSIVE"

## Data-source adaptation: cached scores → `scored_docs`

The original `method.py` builds `scored_docs` by calling the LLM (extraction → gold labelling
→ counterfactual/swap/entrapment decoys → four scoring elicitations). **This is the only part
we replace.** Each row of `mini_demo_data.json` already carries the cached per-item scores
(`z_real`, `z_cf`, `z_swap`, `z_entrapment`) and the crisp gold label, so we simply **regroup
the flat rows back into the per-document `units` structure** the analysis functions expect.
This reproduces exactly the object that `score_units` would have returned — with zero LLM calls.

In [ ]:
def build_scored_docs(examples):
    """Regroup the flat cached-score rows of mini_demo_data.json into the per-document
    {doc_id, k, is_pilot, units:{atomic:[...], bridge:[...]}} structure that the verbatim
    method.py analysis functions consume. Each unit mirrors what score_units() produced:
    z_real / z_cf / z_swap / z_ent + the crisp gold label."""
    docs = {}
    for ex in examples:
        did = ex["metadata_doc_id"]
        if did not in docs:
            docs[did] = {"doc_id": did, "k": ex["metadata_k"], "is_pilot": ex["metadata_is_pilot"],
                         "units": {"atomic": [], "bridge": []}}
        u = {"z_real": ex["metadata_z_real"], "z_cf": ex["metadata_z_cf_decoy"],
             "z_swap": ex["metadata_z_swap_decoy"], "label": ex["metadata_label"]}
        if ex.get("metadata_z_entrapment") is not None:
            u["z_ent"] = ex["metadata_z_entrapment"]
        docs[did]["units"][ex["metadata_family"]].append(u)
    return list(docs.values())


scored_docs = build_scored_docs(examples)
n_units = sum(len(d["units"][f]) for d in scored_docs for f in ("atomic", "bridge"))
print(f"reconstructed {len(scored_docs)} documents, {n_units} scored units")
print("families per doc (first 3):",
      [{f: len(d['units'][f]) for f in ('atomic', 'bridge')} for d in scored_docs[:3]])

## Run the analysis (Stage B: confirmatory diagonal + Stage C: entrapment)

This mirrors the confirmatory portion of `method.py`'s `run()`: build the diagonal for each
family, the decoy-control diagnostics, the populability counts, the entrapment corroboration,
the pre-registered disconfirmation at `alpha*`, and the final calibration verdict.

In [ ]:
diagonal = {fam: diagonal_for_family(scored_docs, fam) for fam in ("bridge", "atomic", "pooled")}
decoy_control = {fam: control_behavior(scored_docs, fam, alpha=0.30) for fam in ("bridge", "atomic")}
pop = populability(scored_docs, ALPHA_STAR)
entrap = {str(a): entrapment_analysis(scored_docs, PRIMARY_FAMILY, a) for a in [ALPHA_STAR, 0.30, 0.50]}
disconf = disconfirmation(scored_docs, PRIMARY_FAMILY, ALPHA_STAR, pop)
verdict = calibration_verdict(diagonal["bridge"], disconf)

print(f"alpha* = {ALPHA_STAR}  primary_family = {PRIMARY_FAMILY}")
print(f"disconfirmation verdict = {disconf['verdict']}")
print(f"CALIBRATION VERDICT     = {verdict}")

## Results — calibration diagonal & the knockoff+ vs plain contrast

We print the bridge-family diagonal table (the same layout as the artifact's `summarize.py`)
and plot **realized FDR vs target `alpha`** for the three gates:

- **knockoff+** (our method) — should hug or stay *below* the `y = alpha` line (conservative),
- **PLAIN** confidence threshold — expected to rise *above* the line (anti-conservative),
- **SWAP** decoy negative control.

The dashed `y = alpha` line is perfect calibration; the shaded band is the knockoff+ doc-block
bootstrap CI.

In [ ]:
def fmt(x, nd=3):
    return "  NA " if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.{nd}f}"

bdiag = diagonal["bridge"]
print("=" * 92)
print(f"BRIDGE diagonal  (n_total={bdiag['n_total']} n_true={bdiag['n_true']} "
      f"n_false={bdiag['n_false_total']} n_pos={bdiag['n_pos']})   [demo subset]")
print("=" * 92)
hdr = f"{'alpha':>6} {'realized':>9} {'ci_lo':>7} {'ci_hi':>7} {'decoy_hat':>9} {'n_adm':>6} {'n_false':>7} {'cert':>5} | {'swap':>6} {'plain':>6} {'plain_est':>9}"
print(hdr)
for r in bdiag["rows"]:
    print(f"{r['target_alpha']:>6} {fmt(r['realized_fdr']):>9} {fmt(r['ci_low']):>7} {fmt(r['ci_high']):>7} "
          f"{fmt(r['decoy_fdr_hat']):>9} {r['n_admitted']:>6} {r['n_false']:>7} {str(r['certified']):>5} | "
          f"{fmt(r['swap_realized_fdr']):>6} {fmt(r['plain_realized_fdr']):>6} {fmt(r['plain_est_fdr']):>9}")

print("\n--- populability (bridge) ---")
print(f"  n_false_total_in_family={pop['bridge']['n_false_total_in_family']}  "
      f"n_admitted_at_alpha*={pop['bridge']['n_admitted_at_alpha_star']}  "
      f"n_false_at_alpha*={pop['bridge']['n_false_at_alpha_star']}")

print("\n--- decoy control (bridge, alpha=0.30) ---")
cb = decoy_control["bridge"]
print(f"  counterfactual_tail_win_rate={fmt(cb['counterfactual_tail_win_rate'])}  "
      f"swap_tail_win_rate={fmt(cb['swap_tail_win_rate'])}")

print("\n--- entrapment corroboration (r=1) ---")
for a, e in entrap.items():
    print(f"  alpha={a}: N_T={e['N_T']} N_E={e['N_E']} combined={fmt(e['fdp_combined'])} "
          f"paired={fmt(e['fdp_paired'])} realized_gold={fmt(e['realized_fdr_gold'])} "
          f"decoy_hat={fmt(e['decoy_fdr_hat'])} agree_realized={e['agree_realized']} agree_decoy={e['agree_decoy']}")

print("\n--- PRE-REGISTERED DISCONFIRMATION ---")
print(f"  family={disconf['family']} alpha*={disconf['alpha_star']} realized={fmt(disconf['realized_fdr'])} "
      f"CI={disconf['ci']}  ->  VERDICT = {disconf['verdict']}")
print(f"  reason: {disconf['reason']}")
print(f"\nOVERALL CALIBRATION VERDICT = {verdict}")

# ---------------------------------------------------------------------------- plot
def _y(v):  # None -> nan so matplotlib skips it
    return np.nan if v is None else v

rows = bdiag["rows"]
xs = [r["target_alpha"] for r in rows]
kk = [_y(r["realized_fdr"]) for r in rows]
lo = [_y(r["ci_low"]) for r in rows]
hi = [_y(r["ci_high"]) for r in rows]
plain = [_y(r["plain_realized_fdr"]) for r in rows]
swap = [_y(r["swap_realized_fdr"]) for r in rows]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot([0, 0.55], [0, 0.55], "k--", lw=1, label="perfect calibration (y = alpha)")
ax.fill_between(xs, lo, hi, color="C0", alpha=0.15, label="knockoff+ bootstrap CI")
ax.plot(xs, kk, "o-", color="C0", lw=2, label="knockoff+ (our method)")
ax.plot(xs, plain, "s--", color="C3", lw=2, label="PLAIN confidence baseline")
ax.plot(xs, swap, "^:", color="C2", lw=2, label="SWAP-decoy control")
ax.set_xlabel("target alpha (FDR budget)")
ax.set_ylabel("realized FDR vs crisp gold")
ax.set_title(f"CLUTRR label-free knockoff+ calibration diagonal\n(bridge family, demo subset, verdict={verdict})")
ax.set_xlim(0, 0.55); ax.set_ylim(0, max(0.65, np.nanmax(plain + kk + [0.3]) + 0.05))
ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("\n[KEY TAKEAWAY] On the demo subset the knockoff+ realized FDR stays flat/conservative "
      "while the PLAIN baseline rises with alpha (anti-conservative) -- the same qualitative "
      "contrast the full 1,540-candidate run certifies as CONFIRMED.")